# Data Generation Pipeline

Stages 1–5 of the benchmarking pipeline.  Run this notebook **once** (in the
`prexsyn_env` kernel) to produce all variant libraries and scoring CSVs that
`pipeline_analysis.ipynb` consumes.

| Stage | Input | Output |
|-------|-------|--------|
| 1 | `chembl_features.npz` | MolSpec objects + feature arrays |
| 2 | Feature arrays | `seeds_for_methods.json` (PrexSyn top-1 per spec) |
| 3 | Seeds | `baseline/baseline_scores.csv` + `synth_checkpoint.json` |
| 4 | Seeds | `crem/crem_scores.csv` + `synth_checkpoint.json` |
| 5 | Seeds | `libinvent/libinvent_scores.csv` + `synth_checkpoint.json` |

All outputs land in `data/generation/`.  Stages 3–5 each end with a
**Stage 4b** cell that applies the property gate first, then runs
AiZynthFinder only on the passing candidates (cheaper than scoring all).

> **Kernel:** `prexsyn_env` (Python 3.11).
> LibINVENT runs as a `conda run -n libinvent_env` subprocess automatically.

In [ ]:
import os, sys
from pathlib import Path

# ── Repo root ──────────────────────────────────────────────────────────────────
ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "src" / "modifications" / "ml_based" / "pipeline"))

# ── Intermediate data directories ──────────────────────────────────────────────
GEN_DIR       = ROOT / "data" / "generation"
BASELINE_DIR  = GEN_DIR / "baseline"
CREM_DIR      = GEN_DIR / "crem"
LI_DIR        = GEN_DIR / "libinvent"
for d in [GEN_DIR, BASELINE_DIR, CREM_DIR, LI_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── Output paths (read by pipeline_analysis.ipynb) ────────────────────────────
SEEDS_JSON       = GEN_DIR  / "seeds_for_methods.json"
BASELINE_SCORES  = BASELINE_DIR / "baseline_scores.csv"
BASELINE_CKPT    = BASELINE_DIR / "synth_checkpoint.json"
CREM_SCORES      = CREM_DIR / "crem_scores.csv"
CREM_CKPT        = CREM_DIR / "synth_checkpoint.json"
LI_SCORES        = LI_DIR  / "libinvent_scores.csv"
LI_CKPT          = LI_DIR  / "synth_checkpoint.json"

# ── Intermediate / scratch paths ──────────────────────────────────────────────
LI_SCAFFOLDS_SMI = LI_DIR  / "scaffolds.smi"
LI_DECORATED_CSV = LI_DIR  / "decorated.csv"
LI_CONFIG_JSON   = LI_DIR  / "decorate_config.json"
LI_LOG_DIR       = LI_DIR  / "logs" / "decorate"
LI_LOG_DIR.mkdir(parents=True, exist_ok=True)

# ── ChEMBL features ───────────────────────────────────────────────────────────
CHEMBL_NPZ = ROOT / "data" / "chembl_features.npz"

# ── External tools ────────────────────────────────────────────────────────────
PREXSYN_URL          = "http://100.65.172.100:8011/sample"
AIZYNTHFINDER_CONFIG = ROOT / "data" / "aizynthfinder" / "config.yml"
CREM_DB              = Path(os.environ.get(
    "CREM_DB_PATH",
    str(ROOT / "data" / "crem_db" / "chembl33_sa2_f5.db")))

LIB_INVENT_ROOT  = ROOT / "src" / "modifications" / "ml_based" / "Lib-INVENT"
# LibINVENT subprocess uses sys.executable (same kernel as this notebook)
LIB_INVENT_MODEL = LIB_INVENT_ROOT / "trained_models" / "reaction_based.model"
LIB_INVENT_PY    = LIB_INVENT_ROOT / "input.py"

# ── Pipeline parameters ───────────────────────────────────────────────────────
N_SPECS            = 100   # ChEMBL molecules used as specs
N_PREXSYN_SAMPLES  = 256   # candidates requested per PrexSyn API call
N_BASELINE_RESAMP  = 128   # extra PrexSyn samples per spec (baseline method)
N_CREM_VARIANTS    = 50    # CReM mutants per seed
N_LI_DECORATIONS   = 32    # LibINVENT decorations per scaffold
N_WORKERS          = 4     # AiZynthFinder parallel workers
SYNTH_TIMEOUT      = 180   # seconds per molecule (0 = no limit)

# ── Scoring thresholds ────────────────────────────────────────────────────────
TAU_T_LIST   = [0.6, 0.7, 0.85]
TAU_D        = 0.8
TAU_T_SCREEN = min(TAU_T_LIST)   # most permissive -- used in Stage 4b screen

# ── Sanity checks ─────────────────────────────────────────────────────────────
print(f"GEN_DIR    : {GEN_DIR}")
print(f"ChEMBL NPZ : {'OK' if CHEMBL_NPZ.exists() else 'MISSING'} ({CHEMBL_NPZ.name})")
print(f"AiZynth    : {'OK' if AIZYNTHFINDER_CONFIG.exists() else 'MISSING'}")
print(f"CReM DB    : {'OK' if CREM_DB.exists() else 'MISSING'} ({CREM_DB.name})")
print(f"LI model   : {'OK' if LIB_INVENT_MODEL.exists() else 'MISSING'}")

In [ ]:
import json, re, subprocess
from concurrent.futures import (
    ProcessPoolExecutor, TimeoutError as FutureTimeoutError, as_completed)
from concurrent.futures.process import BrokenProcessPool

import numpy as np
import pandas as pd
import requests
from rdkit import Chem
from tqdm.notebook import tqdm

from src.evaluation.scoring_v2 import (
    make_spec, score_batch, classify_hits, tanimoto_to_spec)
from src.evaluation.synth_parallel import worker_init, score_one


# ── Stage 4b helper: property gate first, AiZynthFinder second ────────────────
def synth_gate_4b(df_scored: pd.DataFrame, ckpt_path,
                  smiles_col: str = "variant_smiles") -> dict:
    # Property gate first, AiZynthFinder on passing candidates only.
    # Filters to tanimoto >= TAU_T_SCREEN and desirability >= TAU_D,
    # runs retrosynthesis on that subset, writes ckpt_path.
    # Returns {smiles: is_solved}.
    mask = (df_scored["tanimoto"] >= TAU_T_SCREEN) &            (df_scored["desirability"] >= TAU_D)
    candidates = df_scored.loc[mask, smiles_col].dropna().unique().tolist()
    n_all = df_scored[smiles_col].dropna().nunique()
    print(f"[4b] Property gate ({TAU_T_SCREEN}/{TAU_D}): "
          f"{len(candidates):,}/{n_all:,} -> AiZynthFinder")

    if not AIZYNTHFINDER_CONFIG.exists():
        print("     [WARN] Config missing -- marking all synthesizable")
        return {s: True for s in candidates}

    synth: dict = {}
    _pending = list(candidates)
    _to = SYNTH_TIMEOUT or None

    for _round in range(1, 99):
        if _round > 1:
            print(f"     [restart] round {_round}, {len(_pending)} remaining")
        with ProcessPoolExecutor(
            max_workers=N_WORKERS,
            initializer=worker_init,
            initargs=(str(AIZYNTHFINDER_CONFIG),),
            max_tasks_per_child=50,
        ) as pool:
            futures = {pool.submit(score_one, s): s for s in _pending}
            broken  = False
            with tqdm(total=len(_pending),
                      desc=f"AiZynthFinder (round {_round})", unit="mol") as pbar:
                for fut in as_completed(futures):
                    smi = futures[fut]
                    try:
                        _, solved = fut.result(timeout=_to)
                        pbar.update(1)
                    except FutureTimeoutError:
                        solved = False
                    except BrokenProcessPool:
                        broken = True; break
                    except Exception:
                        solved = False
                    synth[smi] = solved
            if broken:
                pass   # outer loop restarts with remaining

        _pending = [s for s in candidates if s not in synth]
        if not _pending:
            break

    with open(ckpt_path, "w") as f:
        json.dump(synth, f)
    n_solved = sum(synth.values())
    print(f"     Solved: {n_solved}/{len(synth)} "
          f"({100 * n_solved / max(len(synth), 1):.1f}%)")
    return synth


print("Imports OK | synth_gate_4b defined")

---
## Stage 1 — ChEMBL Features

Load pre-computed fingerprints and descriptors from `chembl_features.npz`.
Generate the file if missing with:
```
python -m src.generation.main featurize
```

In [ ]:
assert CHEMBL_NPZ.exists(), (
    f"chembl_features.npz not found: {CHEMBL_NPZ}\n"
    "Run: python -m src.generation.main featurize")

_data = np.load(CHEMBL_NPZ, allow_pickle=True)

# Subset to N_SPECS molecules
_idx = np.arange(min(N_SPECS, len(_data["smiles"])))
smiles_arr     = _data["smiles"][_idx]             # (N,)  object
ecfp4_arr      = _data["ecfp4"][_idx]              # (N, 2048)
fcfp4_arr      = _data["fcfp4"][_idx]              # (N, 2048)
rdkit_vals     = _data["rdkit_desc_values"][_idx]  # (N, D)
rdkit_names    = _data["rdkit_desc_names"].tolist()
brics_fps      = _data["brics_fps"][_idx]          # (N, 8, 2048)
brics_exists   = _data["brics_exists"][_idx]       # (N, 8)

# Lookup by SMILES so Stage 2 & 3 are order-independent
feat = {
    smi: {
        "ecfp4":  ecfp4_arr[i],
        "fcfp4":  fcfp4_arr[i],
        "rdkit":  rdkit_vals[i],
        "brics":  brics_fps[i],
        "bric_e": brics_exists[i],
    }
    for i, smi in enumerate(smiles_arr)
}

print(f"Loaded {len(smiles_arr)} molecules from {CHEMBL_NPZ.name}")
print(f"ecfp4={ecfp4_arr.shape}  brics_fps={brics_fps.shape}")

---
## Stage 2 — PrexSyn Seeds

For each spec, POST feature vectors to the PrexSyn API, retrieve
`N_PREXSYN_SAMPLES` candidates, and keep the one with highest ECFP4
Tanimoto to the spec.  Saves `seeds_for_methods.json`.

In [ ]:
def _quality_bin(bq: float) -> str:
    if   bq < 0.50: return "<0.5"
    elif bq < 0.70: return "0.5-0.7"
    elif bq < 0.85: return "0.7-0.85"
    else:           return "0.85-1.0"


if SEEDS_JSON.exists() and SEEDS_JSON.stat().st_size > 0:
    with open(SEEDS_JSON) as f:
        seeds = json.load(f)
    print(f"Loaded {len(seeds)} seeds from cache ({SEEDS_JSON.name})")
else:
    seeds = []
    for smi in tqdm(smiles_arr, desc="PrexSyn seeds"):
        smi = str(smi)
        f   = feat[smi]
        payload = {
            "ecfp4":             f["ecfp4"].tolist(),
            "fcfp4":             f["fcfp4"].tolist(),
            "rdkit_desc_values": f["rdkit"].tolist(),
            "rdkit_desc_names":  rdkit_names,
            "brics_fps":         f["brics"].tolist(),
            "brics_exists":      f["bric_e"].tolist(),
            "source_smiles":     smi,
            "num_samples":       N_PREXSYN_SAMPLES,
        }
        try:
            resp       = requests.post(PREXSYN_URL, json=payload, timeout=300)
            resp.raise_for_status()
            candidates = resp.json().get("generated_smiles", [])
        except Exception as e:
            print(f"  [WARN] {smi[:40]}: {e}")
            candidates = []

        spec = make_spec(smi)
        best_smi, best_t = smi, 0.0
        if spec and candidates:
            for c in candidates:
                if "." in c:
                    continue
                mol = Chem.MolFromSmiles(c)
                if mol:
                    t = tanimoto_to_spec(mol, spec)
                    if t > best_t:
                        best_t, best_smi = t, c

        seeds.append({
            "spec_smiles":      smi,
            "seed_smiles":      best_smi,
            "baseline_quality": round(best_t, 4),
            "quality_bin":      _quality_bin(best_t),
            "methods":          {},
        })

    with open(SEEDS_JSON, "w") as f:
        json.dump(seeds, f, indent=2)
    print(f"Seeds saved -> {SEEDS_JSON}")

# Build lookup maps used by all downstream stages
spec_to_seed = {e["spec_smiles"]: e["seed_smiles"]      for e in seeds}
spec_to_bq   = {e["spec_smiles"]: e["baseline_quality"] for e in seeds}
specs_list   = [e["spec_smiles"] for e in seeds]
seeds_list   = [e["seed_smiles"] for e in seeds]

from collections import Counter
print(Counter(e["quality_bin"] for e in seeds))

---
## Stage 3 — PrexSyn Resampling (Baseline)

Call PrexSyn again for `N_BASELINE_RESAMP` samples per spec.  Score each
variant with `scoring_v2.score_batch()`.  Then apply the Stage 4b gate.

In [ ]:
if BASELINE_SCORES.exists() and BASELINE_SCORES.stat().st_size > 0:
    df_baseline = pd.read_csv(BASELINE_SCORES)
    print(f"Loaded {len(df_baseline):,} rows from cache ({BASELINE_SCORES.name})")
else:
    _rows = []
    for smi in tqdm(specs_list, desc="Baseline resample"):
        f    = feat.get(smi)
        spec = make_spec(smi)
        if f is None or spec is None:
            continue
        payload = {
            "ecfp4":             f["ecfp4"].tolist(),
            "fcfp4":             f["fcfp4"].tolist(),
            "rdkit_desc_values": f["rdkit"].tolist(),
            "rdkit_desc_names":  rdkit_names,
            "brics_fps":         f["brics"].tolist(),
            "brics_exists":      f["bric_e"].tolist(),
            "source_smiles":     smi,
            "num_samples":       N_BASELINE_RESAMP,
        }
        try:
            resp     = requests.post(PREXSYN_URL, json=payload, timeout=300)
            resp.raise_for_status()
            variants = resp.json().get("generated_smiles", [])
        except Exception as e:
            print(f"  [WARN] {smi[:40]}: {e}")
            variants = []

        _rows.append(score_batch(
            variants, spec,
            baseline_quality=spec_to_bq[smi],
            method="baseline",
        ))

    df_baseline = pd.concat(_rows, ignore_index=True) if _rows else pd.DataFrame()
    df_baseline.to_csv(BASELINE_SCORES, index=False)
    print(f"Saved -> {BASELINE_SCORES}  ({len(df_baseline):,} rows)")

In [ ]:
# Stage 4b gate for baseline: score first, AiZynthFinder on property-passing only
synth_baseline = synth_gate_4b(df_baseline, BASELINE_CKPT)
df_baseline["is_synth"] = (
    df_baseline["variant_smiles"].map(synth_baseline).fillna(False))
_hits = classify_hits(df_baseline, TAU_T_LIST[0], TAU_D) & df_baseline["is_synth"]
print(f"Baseline hits (tau_t={TAU_T_LIST[0]}, tau_d={TAU_D}): "
      f"{_hits.sum()} / {len(df_baseline)}")

---
## Stage 4 — CReM Modification

Apply context-dependent fragment substitution (CReM) to each seed.
Requires `replacements02_sc2.db` (set `CREM_DB_PATH` or update `CREM_DB` above).
Download from: <https://zenodo.org/record/4519690>

In [ ]:
from src.modifications.rule_based.crem_modifier import CRemModifier

if CREM_SCORES.exists() and CREM_SCORES.stat().st_size > 0:
    df_crem = pd.read_csv(CREM_SCORES)
    print(f"Loaded {len(df_crem):,} rows from cache ({CREM_SCORES.name})")
else:
    if not CREM_DB.exists():
        raise FileNotFoundError(
            f"CReM database not found: {CREM_DB}\n"
            "Download chembl33_sa2_f5.db from https://zenodo.org/records/16909329")

    crem = CRemModifier(db_path=CREM_DB)
    _rows = []
    for spec_smi, seed_smi in tqdm(
            list(zip(specs_list, seeds_list)), desc="CReM"):
        spec = make_spec(spec_smi)
        if spec is None:
            continue
        try:
            variants = crem.modify(seed_smi, N_CREM_VARIANTS)
        except Exception as e:
            print(f"  [WARN] {seed_smi[:40]}: {e}")
            variants = []
        _rows.append(score_batch(
            variants, spec,
            baseline_quality=spec_to_bq[spec_smi],
            method="CReM",
        ))

    df_crem = pd.concat(_rows, ignore_index=True) if _rows else pd.DataFrame()
    df_crem.to_csv(CREM_SCORES, index=False)
    print(f"Saved -> {CREM_SCORES}  ({len(df_crem):,} rows)")

In [ ]:
# Stage 4b gate for CReM
synth_crem = synth_gate_4b(df_crem, CREM_CKPT)
df_crem["is_synth"] = df_crem["variant_smiles"].map(synth_crem).fillna(False)
_hits = classify_hits(df_crem, TAU_T_LIST[0], TAU_D) & df_crem["is_synth"]
print(f"CReM hits (tau_t={TAU_T_LIST[0]}, tau_d={TAU_D}): "
      f"{_hits.sum()} / {len(df_crem)}")

---
## Stage 5 — LibINVENT Decoration

Decompose each seed into BRICS scaffolds, then decorate with the
pretrained LibINVENT RNN (runs as a subprocess in `libinvent_env`).

In [ ]:
from fragment import get_scaffolds

# Extract BRICS scaffolds from each seed
scaffold_map: dict[str, list[str]] = {}   # spec_smi -> [scaffold_smi, ...]
_all_sc: list[str] = []
_seen   = set()

for spec_smi, seed_smi in tqdm(
        list(zip(specs_list, seeds_list)), desc="BRICS scaffolds"):
    scs = get_scaffolds(seed_smi, method="brics")
    scaffold_map[spec_smi] = scs
    for sc in scs:
        if sc not in _seen:
            _seen.add(sc)
            _all_sc.append(sc)

print(f"Seeds: {len(specs_list)} | Unique scaffolds: {len(_all_sc)}")

In [ ]:
from configs import make_decorate_config

# Scaffold canonicalization -- BRICS labels [*:0],[*:1]; LibINVENT outputs [*]
_ATTACH = re.compile(r'\[\*(?::\d+)?\]')

def _canon_sc(smi: str):
    mol = Chem.MolFromSmiles(re.sub(r'\[\*(?::\d+)?\]', '[*]', smi))
    if mol is None:
        return None
    Chem.RemoveStereochemistry(mol)
    return Chem.MolToSmiles(mol)


# Build canonical scaffold -> [spec_smi, ...] reverse map
_sc_to_specs: dict[str, list[str]] = {}
for spec_smi, scs in scaffold_map.items():
    for sc in scs:
        key = _canon_sc(sc)
        if key:
            _sc_to_specs.setdefault(key, []).append(spec_smi)

# ── Run LibINVENT if output not cached ────────────────────────────────────────
if LI_SCORES.exists() and LI_SCORES.stat().st_size > 0:
    df_li = pd.read_csv(LI_SCORES)
    print(f"Loaded {len(df_li):,} rows from cache ({LI_SCORES.name})")
else:
    assert LIB_INVENT_MODEL.exists(), f"Model not found: {LIB_INVENT_MODEL}"
    assert LIB_INVENT_PY.exists(),    f"input.py not found: {LIB_INVENT_PY}"

    # Write scaffolds.smi
    LI_SCAFFOLDS_SMI.write_text("\n".join(_all_sc))
    print(f"Wrote {len(_all_sc)} scaffolds -> {LI_SCAFFOLDS_SMI}")

    # Write decorate_config.json
    cfg = make_decorate_config(
        model_path         = str(LIB_INVENT_MODEL.resolve()),
        scaffolds_smi_path = str(LI_SCAFFOLDS_SMI.resolve()),
        output_csv_path    = str(LI_DECORATED_CSV.resolve()),
        logging_path       = str(LI_LOG_DIR.resolve()),
        batch_size         = 64,
        n_decorations      = N_LI_DECORATIONS,
    )
    LI_CONFIG_JSON.write_text(json.dumps(cfg, indent=2))

    # Run LibINVENT with the current kernel's Python (same env as this notebook)
    import sys as _sys
    cmd = [_sys.executable,
           str(LIB_INVENT_PY.resolve()),
           str(LI_CONFIG_JSON.resolve())]
    print(f"Running: {' '.join(cmd)}\n")
    result = subprocess.run(
        cmd, cwd=str(LIB_INVENT_ROOT),
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    print(result.stdout or "")
    if result.returncode != 0:
        raise RuntimeError(
            f"LibINVENT exited with code {result.returncode}")

    # Parse decorated.csv and score
    assert LI_DECORATED_CSV.exists(), "decorated.csv not produced by LibINVENT"
    df_dec = pd.read_csv(LI_DECORATED_CSV)
    df_dec = df_dec[df_dec["SMILES"].apply(
        lambda s: Chem.MolFromSmiles(str(s)) is not None
    )].copy()
    df_dec["_canon_sc"] = df_dec["Scaffold"].apply(_canon_sc)

    # Score each variant against its matching spec(s)
    _nll_map = dict(zip(df_dec["SMILES"], df_dec.get("Likelihoods", [None]*len(df_dec))))
    _rows = []
    for spec_smi in tqdm(specs_list, desc="Score LibINVENT"):
        spec = make_spec(spec_smi)
        if spec is None:
            continue
        # Variants whose scaffold came from this spec's seed
        _my_scs = {_canon_sc(sc) for sc in scaffold_map.get(spec_smi, [])}
        _mask   = df_dec["_canon_sc"].isin(_my_scs)
        variants = df_dec.loc[_mask, "SMILES"].tolist()
        if not variants:
            continue
        df_s = score_batch(variants, spec,
                           baseline_quality=spec_to_bq[spec_smi],
                           method="LibINVENT")
        df_s["nll"] = df_s["variant_smiles"].map(_nll_map)
        _rows.append(df_s)

    df_li = pd.concat(_rows, ignore_index=True) if _rows else pd.DataFrame()
    df_li.to_csv(LI_SCORES, index=False)
    print(f"Saved -> {LI_SCORES}  ({len(df_li):,} rows)")

In [ ]:
# Stage 4b gate for LibINVENT
synth_li = synth_gate_4b(df_li, LI_CKPT)
df_li["is_synth"] = df_li["variant_smiles"].map(synth_li).fillna(False)
_hits = classify_hits(df_li, TAU_T_LIST[0], TAU_D) & df_li["is_synth"]
print(f"LibINVENT hits (tau_t={TAU_T_LIST[0]}, tau_d={TAU_D}): "
      f"{_hits.sum()} / {len(df_li)}")

---
## Outputs

All files consumed by `pipeline_analysis.ipynb`.

In [ ]:
outputs = [
    ("seeds_for_methods.json", SEEDS_JSON),
    ("baseline_scores.csv",    BASELINE_SCORES),
    ("baseline ckpt",          BASELINE_CKPT),
    ("crem_scores.csv",        CREM_SCORES),
    ("crem ckpt",              CREM_CKPT),
    ("libinvent_scores.csv",   LI_SCORES),
    ("libinvent ckpt",         LI_CKPT),
]
print(f"{'File':<28}  {'Size':>8}  {'Status'}")
print("-" * 55)
for label, path in outputs:
    ok   = path.exists()
    size = f"{path.stat().st_size/1e3:.1f} KB" if ok else "-"
    print(f"{label:<28}  {size:>8}  {'OK' if ok else 'MISSING'}")

print()
print("Add to pipeline_analysis.ipynb METHODS dict:")
print(f'  "baseline" : {{')
print(f'      "scores":     DATA_DIR / "generation/baseline/baseline_scores.csv",')
print(f'      "synth_ckpt": DATA_DIR / "generation/baseline/synth_checkpoint.json",')
print(f'  }},')
print(f'  "CReM" : {{')
print(f'      "scores":     DATA_DIR / "generation/crem/crem_scores.csv",')
print(f'      "synth_ckpt": DATA_DIR / "generation/crem/synth_checkpoint.json",')
print(f'  }},')
print(f'  "LibINVENT" : {{')
print(f'      "scores":     DATA_DIR / "generation/libinvent/libinvent_scores.csv",')
print(f'      "synth_ckpt": DATA_DIR / "generation/libinvent/synth_checkpoint.json",')
print(f'  }},')